# Lab 1 : Embeddings are just numbers

*W3 RAG Part 1 · Utrains LLMOps 8-Week Course*

Run each cell in order. Read the output. Move to the next.

See the matching slide in this week's concepts deck for the real-world story this lab teaches.

## What we are achieving in this lab

**Objective.** Know what an embedding *is* and how to *implement* one with LangChain. By the end you can look at a vector, name the model that produced it, and explain `embed_query` vs `embed_documents`.

**Prerequisites.** Weeks 1 and 2 finished. An OpenAI API key in the repository-root `.env`. See [README.md](./README.md).

**What this lab uses.**

| Layer | What it does | What we use | Why this one |
|-------|----------------|-------------|--------------|
| Embedding model | Turns text into a vector | OpenAI `text-embedding-3-small` | Cheapest current OpenAI embedder. ~$0.02 / 1M tokens. |
| Library | One interface for any embedding vendor | **LangChain** (`OpenAIEmbeddings`) | Swap OpenAI for Voyage or Azure later without rewriting retrieval. |

LangGraph and Claude come later, once Lab 2 has given you a way to measure "close." This lab stops at the vector.

**What you will do.**

1. Load the OpenAI key from the environment (not hardcoded).
2. Embed one support question with LangChain's `OpenAIEmbeddings` and look at the numbers.
3. See that a word and a paragraph become vectors of the **same length**.
4. Learn `embed_query` vs `embed_documents`, and the other embedding model ids in the table below.
5. Compare two similar sentences and one unrelated sentence by looking at the first few numbers.

**What you should see.**

- A vector of **1536** floats from `text-embedding-3-small`.
- Similar questions producing numbers that look closer than an unrelated question.

**Cost.** A handful of embedding calls. Fractions of a cent.

## Where this sits in LLMOps

Week 1: can I get an answer at all?
Week 2: can I trust that answer enough to hand it to another system?
**Week 3: can I make it answer about *my* data?**

A chat model predicts the next token. It does not look up your handbook. To give it your data you **index** that data as vectors, **retrieve** the nearest ones, then **generate** an answer that is only allowed to use those passages. That pattern is RAG.

This lab is the first piece: what a vector is, and how production code produces one. Lab 2 measures "close." Later labs retrieve and then generate.

### A fact interviews often test

**Anthropic does not offer an embedding model.** Claude writes answers. It does not turn text into vectors. Anthropic's own docs say so, and they point teams to [Voyage AI](https://platform.claude.com/docs/en/build-with-claude/embeddings) as a preferred embedding partner.

So a Claude RAG app is almost always:

`embedding provider` (OpenAI, Voyage, Cohere, Bedrock, Azure)  +  `Claude`  +  `vector store`  +  `LangChain`

If someone asks "which Anthropic embedding model did you use?" the correct answer is: **there isn't one.** Then name the embedder you *did* use, and why.

## Embedding models you should be able to name

You will implement **one** cheap production model below. You should still be able to **name the others**. Prices and ids change; the shape of this table is what matters.

| Vendor | Model id | Typical dim | When teams pick it |
|--------|------------------------|-------------|------------------------|
| **OpenAI** | `text-embedding-3-small` | 1536 | Default in a huge share of LangChain RAG codebases. Cheap. This lab. |
| **OpenAI** | `text-embedding-3-large` | 3072 (truncatable) | Higher recall; legal / medical / "wrong retrieval is expensive." ~6× the small model's price. |
| **OpenAI** | `text-embedding-ada-002` | 1536 | Legacy. Still in old indexes. Do not start a new project on it. |
| **Azure OpenAI** | same `text-embedding-3-*` ids, Azure endpoint | 1536 / 3072 | Enterprises that already live on Azure and Entra ID. Same vectors, different bill and data residency. |
| **Voyage AI** (Anthropic's recommended partner) | `voyage-4-lite` | 1024 (256/512/2048) | Cheapest Voyage 4. Latency and cost. |
| **Voyage AI** | `voyage-4` | 1024 | Balanced default when the generator is Claude. |
| **Voyage AI** | `voyage-4-large` | 1024 | Best Voyage retrieval quality. |
| **Voyage AI** | `voyage-code-3`, `voyage-law-2`, `voyage-finance-2` | 1024 | Domain corpora (code, legal, finance). |
| **Cohere** | `embed-v4.0` / `embed-english-v3.0` | 1024 | Strong multilingual + rerank combo. Common on Bedrock. |
| **Google** | `gemini-embedding-001` | 768+ | GCP-native stacks, Vertex AI. |
| **AWS Bedrock** | `amazon.titan-embed-text-v2:0` | 1024 (variable) | AWS-only procurement. No extra vendor contract. |
| **Hugging Face / self-host** | `BAAI/bge-m3`, E5, GTE | varies | Air-gapped, on-prem, or "data never leaves the VPC." |

Two rules that apply no matter which vendor you pick:

1. **One index, one embedding model.** An index is just a stored list of vectors. Those vectors only mean something *next to other vectors from the same model*. OpenAI's `text-embedding-3-small` produces 1536 numbers. Voyage's `voyage-4` produces 1024. If you drop both into the same list and search, the math is comparing apples to oranges. Search will look like it works and return junk. If you change the model, you must embed every document again.

2. **Write down which model you used.** Store the model name (for example `text-embedding-3-small`) and the number of dimensions (1536) next to the index. If someone later switches the model in code but forgets to re-embed the documents, nothing crashes. There is no error. Answers just get worse, and you will not know why until you check that log.

### Step 1. Keys, the way a service loads them

Copy `.env.example` to `.env` at the **repository root** if you have not already. Paste your OpenAI key. Never commit `.env`.

In production these same secrets live in AWS Secrets Manager, Doppler, or HashiCorp Vault (Week 6). `load_dotenv()` is the local stand-in.

In [2]:
import os
from pathlib import Path

from dotenv import load_dotenv, find_dotenv

# Walks up from the notebook / cwd until it finds the repo-root .env.
load_dotenv(find_dotenv(usecwd=True))

required = ["OPENAI_API_KEY"]
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise EnvironmentError(
        "Missing OPENAI_API_KEY. "
        "Copy .env.example to .env at the repository root, paste the key, "
        "and restart the kernel."
    )

print("OPENAI_API_KEY : set")
print("cwd            :", Path.cwd())

OPENAI_API_KEY    : set
ANTHROPIC_API_KEY : set
cwd               : h:\Project\llmops-course\week03


### Step 2. Implement an embedding: LangChain + OpenAI

Application code rarely calls `openai.embeddings.create` by hand. It goes through **LangChain's `Embeddings` interface**, so swapping OpenAI for Voyage or Azure is a constructor change, not a rewrite of retrieval.

Current import:

```python
from langchain_openai import OpenAIEmbeddings
```

We use `text-embedding-3-small`: the cheapest OpenAI embedder that is still a current production model. That is the same habit as Week 2 picking Haiku over Opus — **cheapest model that clears the bar.**

In [3]:
from langchain_openai import OpenAIEmbeddings

# Production default for this course. Cheap, current, 1536 dimensions.
EMBED_MODEL = "text-embedding-3-small"

# --- Interview crib: constructors you should be able to write on a whiteboard ---
# from langchain_openai import OpenAIEmbeddings
# from langchain_voyageai import VoyageAIEmbeddings
# from langchain_cohere import CohereEmbeddings
#
# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")          # this lab
# embeddings = OpenAIEmbeddings(model="text-embedding-3-large")          # higher recall, ~6x price
# embeddings = OpenAIEmbeddings(                                         # Azure OpenAI v1 endpoint
#     model="text-embedding-3-small",
#     base_url="https://YOUR-RESOURCE.openai.azure.com/openai/v1/",
# )
# embeddings = VoyageAIEmbeddings(model="voyage-4-lite")                 # Anthropic-recommended partner, cheapest Voyage 4
# embeddings = VoyageAIEmbeddings(model="voyage-4")                      # Claude-stack default
# embeddings = CohereEmbeddings(model="embed-v4.0")                      # common on AWS Bedrock
# # Google Vertex: langchain-google-genai  /  Bedrock Titan: langchain-aws
# # Never put two of these into the same vector index.

embeddings = OpenAIEmbeddings(model=EMBED_MODEL)

text = "How do I reset my password?"
vec = embeddings.embed_query(text)

print("--- 1.A : one sentence, as numbers ---")
print(f"text        : {text!r}")
print(f"model       : {EMBED_MODEL}")
print(f"class       : {type(embeddings).__name__}")
print(f"type        : {type(vec).__name__}")
print(f"length      : {len(vec)} numbers")
print(f"first 8     : {[round(x, 4) for x in vec[:8]]}")
print(f"last 8      : {[round(x, 4) for x in vec[-8:]]}")
print(f"min / max   : {min(vec):.4f}  /  {max(vec):.4f}")

--- 1.A : one sentence, as numbers ---
text        : 'How do I reset my password?'
model       : text-embedding-3-small
class       : OpenAIEmbeddings
type        : list
length      : 1536 numbers
first 8     : [0.0176, -0.0457, 0.0298, 0.0219, -0.0486, 0.002, 0.0016, 0.0614]
last 8      : [-0.0197, 0.012, -0.0021, 0.0352, -0.0285, 0.0001, -0.0263, 0.0124]
min / max   : -0.0958  /  0.1183


> **The "oh!" moment.** There is no hidden object. An embedding is a Python list of floats. For `text-embedding-3-small` that list is **1536** long. `text-embedding-3-large` is 3072. Voyage 4 defaults to 1024. The length is a **property of the model you chose**, the same way context window was a property of the chat model in Week 1.
>
> You will never read all 1536 numbers in production. You *should* print the length, a slice, and the model id once, so you know what you are storing.

### Step 3. Same shape, any length of text

The word `"password"` and a whole paragraph both come back as a list of **1536** numbers. The text can be short or long. The vector length does not change.

That shared length is what lets you search.

**Nearest neighbour**, in plain language: you have many stored vectors (one per document). A new question also becomes a vector. "Search" means: walk the list and pick the stored vector that sits **closest** to the question vector. Closest here is a distance in that 1536-number space, not "same words." Lab 2 will measure that distance. Today you only need the idea.

You can only do that if every vector has the **same number of slots**. Comparing a list of 10 numbers to a list of 1536 numbers has no fair distance: the two objects are not the same shape. The embedding model's job is to squash any text into that one fixed shape so the comparison is always legal.

In [4]:
short = "password"
long = (
    "To reset your password, open the sign-in page, click Forgot Password, "
    "and check the email we send within five minutes. If it is not there, "
    "look in spam. You can also ask your manager to trigger a reset from "
    "the admin portal under Account > Security."
)

short_vec = embeddings.embed_query(short)
long_vec = embeddings.embed_query(long)

print(f"{short!r:12} -> {len(short_vec)} numbers")
print(f"{'paragraph':12} -> {len(long_vec)} numbers")
print("same length :", len(short_vec) == len(long_vec))

'password'   -> 1536 numbers
paragraph    -> 1536 numbers
same length : True


### Step 4. `embed_query` vs `embed_documents`

RAG has two moments in time. Each moment uses a different method.

**Moment 1 — you build the index (once, when documents change).**
You split the handbook into chunks. You turn every chunk into a vector and store those vectors. That call is `embed_documents`. It takes a **list** of strings and returns a **list** of vectors, one per chunk.

**Moment 2 — a user asks a question (every request).**
You turn that one question into a vector, then search for the closest stored chunk vectors. That call is `embed_query`. It takes **one** string and returns **one** vector.

| | `embed_documents` | `embed_query` |
|---|---|---|
| When | Index time | Question time |
| Input | Many strings (your chunks) | One string (the question) |
| Output | Many vectors | One vector |
| How often | Rare: when you add or change docs | Every search |

Think of it as packing boxes vs asking for a box. You pack the warehouse with `embed_documents`. You hold up one item and ask "which stored box is nearest?" with `embed_query`.

**Are the vectors different?**

The **length** is the same in both cases: still 1536 numbers for `text-embedding-3-small`. Same model, same shape.

The **numbers inside** depend on the vendor:

- **This lab (OpenAI v3).** Query and document live in one shared space. If you embed the exact same sentence with `embed_query` and with `embed_documents`, you should get (almost) the same vector. LangChain still gives you two methods because one is "a list of chunks" and the other is "one question" — that is how RAG is written, and it matches other vendors.
- **Voyage and Cohere.** These models were trained with two roles. A question is marked `input_type="query"`. A stored chunk is marked `input_type="document"`. The **same sentence** can then produce **two slightly different vectors**, one for each role. That is on purpose: it often makes "question vs paragraph" search more accurate. LangChain's `embed_query` / `embed_documents` split is how those libraries send the right role without you remembering the flag every time.

**Rule of thumb.** Never embed your handbook with `embed_query` in a loop if you can batch it. Never embed the user's question with `embed_documents` unless you wrap it in a one-item list. Use each method for the moment it names.

The next cell shows both calls on this OpenAI model. Compare the counts: one vector vs two vectors, each 1536 long.

In [5]:
query_vec = embeddings.embed_query(text)
doc_vecs = embeddings.embed_documents(
    [
        "Click Forgot Password on the sign-in page.",
        "Our offices open at 9am Pacific.",
    ]
)

print("--- 1.B : the two methods RAG actually calls ---")
print(f"embed_query    : 1 vector of {len(query_vec)} numbers")
print(f"embed_documents: {len(doc_vecs)} vectors, each {len(doc_vecs[0])} numbers")

--- 1.B : the two methods RAG actually calls ---
embed_query    : 1 vector of 1536 numbers
embed_documents: 2 vectors, each 1536 numbers


### Step 5. Similar meaning should land near similar numbers

Do not compute a formula yet. Just look. Two paraphrases of a password-reset question, plus an unrelated office-hours question.

You are training your eye for Lab 2. If the first eight numbers of the two password questions look closer to each other than to the office-hours question, the model is doing its job.

In [6]:
# Three sentences. The first two mean the same thing. The third does not.
sentence_a = "How do I reset my password?"
sentence_b = "I forgot my password, can you help?"
sentence_c = "What time does the office open?"

# Turn each sentence into a vector (a list of 1536 numbers).
vector_a = embeddings.embed_query(sentence_a)
vector_b = embeddings.embed_query(sentence_b)
vector_c = embeddings.embed_query(sentence_c)

# 1536 numbers is too many to read, so we only show the first 8.
# [:8] means "the first eight items". round(..., 3) shortens each number.
print("A  reset password :", [round(n, 3) for n in vector_a[:8]])
print("B  forgot password:", [round(n, 3) for n in vector_b[:8]])
print("C  office hours   :", [round(n, 3) for n in vector_c[:8]])

sentence                                          first 8 numbers
------------------------------------------------------------------------------------------
How do I reset my password?                       [ 0.018, -0.046,  0.030,  0.022, -0.049,  0.002,  0.002,  0.061]
I forgot my password, can you help?               [ 0.019,  0.001,  0.002, -0.007, -0.033, -0.019, -0.016,  0.051]
What time does the office open?                   [-0.008,  0.089,  0.032, -0.008, -0.013,  0.019,  0.060,  0.055]


The first two rows should look more alike than either does to the third. Your eye is a bad measuring tool past about four dimensions, which is why Lab 2 exists. The idea to keep: **meaning became geometry.**

## What you should be able to explain

Write these down. They are the Lab 1 check.

> "An embedding is a fixed-length vector. I pick a model, I log its name and dimension, and I never mix vectors from two models in the same index."

> "Claude does not embed. In a Claude app I still choose an embedder: OpenAI `text-embedding-3-small` when cost is the constraint, `text-embedding-3-large` or Voyage `voyage-4` when retrieval quality is the constraint. Anthropic recommends Voyage; most LangChain codebases I have seen still start on OpenAI."

> "In LangChain I use `embed_documents` at index time and `embed_query` at question time."

> "I pick the cheapest model that clears the eval, not the most famous one. For this lab that is `text-embedding-3-small`."

**Lab 2** takes the 1536-number lists you just printed and measures "close" with one formula. That is cosine similarity. LangGraph (retrieve, then generate) waits until you have that formula.